In [9]:
#%pip install gymnasium mujoco stable-baselines3 torch tensorboard imageio jupyter

In [10]:
import gymnasium as gym
import mujoco
from gymnasium.envs.registration import register
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
from stable_baselines3 import SAC, PPO
from stable_baselines3.common.callbacks import EvalCallback

In [11]:
class RacingAntEnv(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        self.goal_distance = 100.0
        self.max_steps = 60
        self.elapsed_steps = 0
        self.prev_x_pos = 0.0

        self.env = gym.make('Ant-v5', render_mode=render_mode)
        # 👇 Guardamos una referencia al entorno sin envolturas
        self.unwrapped_env = self.env.unwrapped

        orig_obs_space = self.env.observation_space
        low = np.concatenate(([-np.inf, -np.inf], orig_obs_space.low))
        high = np.concatenate(([np.inf, np.inf], orig_obs_space.high))
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)
        self.action_space = self.env.action_space

    def _get_obs(self):
        """Obtiene la observación: [pos_x, vel_x, obs_original]."""
        # Usamos unwrapped_env para acceder a los datos internos
        x_position = self.unwrapped_env.data.qpos[0]
        x_velocity = self.unwrapped_env.data.qvel[0]
        original_obs = self.unwrapped_env._get_obs()   # ✅ Ahora sí funciona
        return np.concatenate(([x_position, x_velocity], original_obs)).astype(np.float32)

    def step(self, action):
        obs_original, reward_original, terminated, truncated, info = self.env.step(action)
        self.elapsed_steps += 1

        # Usamos unwrapped_env para la posición/velocidad
        x_position = self.unwrapped_env.data.qpos[0]
        x_velocity = self.unwrapped_env.data.qvel[0]
        done = terminated or truncated or self.elapsed_steps >= self.max_steps

        # Recompensa personalizada
        reward = 0.0
        distance_moved = x_position - self.prev_x_pos
        reward += distance_moved * 1.0
        reward -= 0.1

        if x_position >= self.goal_distance:
            print(f"🎉 Meta alcanzada en el paso {self.elapsed_steps}!")
            reward += 100.0
            done = True

        self.prev_x_pos = x_position
        obs = self._get_obs()
        return obs, reward, done, False, info

    def reset(self, *, seed=None, options=None):
        self.elapsed_steps = 0
        self.prev_x_pos = 0.0
        obs_original, info = self.env.reset(seed=seed, options=options)
        obs = self._get_obs()
        return obs, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()
    
register(
    id='RacingAnt-v0',
    entry_point='__main__:RacingAntEnv',
)

In [12]:
env = gym.make('RacingAnt-v0', render_mode='human')
obs, _ = env.reset()
print("Observación inicial:", obs.shape)
for _ in range(10):
    action = env.action_space.sample()
    obs, reward, done, _, _ = env.step(action)
    print(f"Reward: {reward:.2f}, Posición X: {obs[0]:.2f}")
    if done:
        break
env.close()

c:\Users\Carla\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='human' that is not in the possible render_modes ([]).
  logger.warn(
c:\Users\Carla\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\Carla\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Observación inicial: (107,)
Reward: -0.10, Posición X: 0.00
Reward: -0.11, Posición X: -0.01
Reward: -0.10, Posición X: -0.01
Reward: -0.08, Posición X: 0.01
Reward: -0.10, Posición X: 0.01
Reward: -0.10, Posición X: 0.01
Reward: -0.10, Posición X: 0.00
Reward: -0.10, Posición X: -0.00
Reward: -0.08, Posición X: 0.02
Reward: -0.08, Posición X: 0.04


In [16]:
import imageio
import numpy as np
from IPython.display import Image, display

env = RacingAntEnv(render_mode='rgb_array')
obs, _ = env.reset()
model = PPO.load("modelo_carrera")
frames = []

for _ in range(2000):
    action, _ = model.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = env.step(action)
    frame = env.render()
    frames.append(frame)
    if terminated or truncated:
        break
env.close()

# Guardar como GIF
gif_path = "carrera_entrenada.gif"
imageio.mimsave(gif_path, frames, fps=30)
print(f"GIF guardado en {gif_path}")

# Mostrar en el notebook
display(Image(filename=gif_path))

FileNotFoundError: [Errno 2] No such file or directory: 'modelo_carrera.zip'

In [13]:
from stable_baselines3 import PPO

# Creamos el entorno de entrenamiento (sin renderizar para ir más rápido)
train_env = gym.make('RacingAnt-v0')

# Creamos el modelo con una política MLP
model = PPO('MlpPolicy', train_env, verbose=1)

# Entrenamos durante 1,000,000 de pasos ambientales
model.learn(total_timesteps=1_000_000)

# Guardamos el modelo entrenado para no perder el progreso
model.save("racing_ant")

print("Entrenamiento completado ✅")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 42       |
|    ep_rew_mean     | -4.39    |
| time/              |          |
|    fps             | 2003     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 41.1        |
|    ep_rew_mean          | -4.34       |
| time/                   |             |
|    fps                  | 1558        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.012189426 |
|    clip_fraction        | 0.139       |
|    clip_range           | 0.2         |
|    entropy_loss   

In [14]:
import time

env = gym.make('RacingAnt-v0', render_mode='human')
obs, _ = env.reset()
model = PPO.load("racing_ant")

# Un bucle de hasta 1000 pasos, pero el agente debería terminar antes.
for _ in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, _, _ = env.step(action)
    time.sleep(0.01) # Pequeña pausa para que la visualización sea fluida
    if terminated:
        print("🏁 Agente terminó la carrera o el tiempo máximo.")
        break
env.close()

c:\Users\Carla\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='human' that is not in the possible render_modes ([]).
  logger.warn(
c:\Users\Carla\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\Carla\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


🏁 Agente terminó la carrera o el tiempo máximo.


In [5]:
# ============================================================
# CORREGIDO: ENTORNO CON dt BIEN DEFINIDO (100m en 60s reales)
# ============================================================

# !pip install gymnasium stable-baselines3 mujoco imageio[ffmpeg] torch

import gymnasium as gym
import numpy as np
from gymnasium.envs.registration import register
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback
import imageio
import os
import torch
import shutil

# --- Entorno corregido: cálculo de dt desde el entorno desempaquetado ---
class RacingAntEnv(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        self.goal_distance = 100.0          # metros

        # Creamos el entorno base (aunque esté envuelto en TimeLimit)
        self.env = gym.make('Ant-v5', render_mode=render_mode)
        self.unwrapped_env = self.env.unwrapped   # quitamos TimeLimit

        # Ahora dt está disponible en el entorno desempaquetado
        self.dt = self.unwrapped_env.dt      # 0.01 seg/paso por defecto
        self.max_steps = int(60.0 / self.dt)   # 6000 pasos para 60 segundos
        print(f"dt={self.dt} seg/paso → {self.max_steps} pasos para 60 segundos")

        self.elapsed_steps = 0
        self.prev_x_pos = 0.0

        # Espacios de observación y acción
        orig_obs_space = self.env.observation_space
        low = np.concatenate(([-np.inf, -np.inf], orig_obs_space.low))
        high = np.concatenate(([np.inf, np.inf], orig_obs_space.high))
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)
        self.action_space = self.env.action_space

    def _get_obs(self):
        x_pos = self.unwrapped_env.data.qpos[0]
        x_vel = self.unwrapped_env.data.qvel[0]
        obs_original = self.unwrapped_env._get_obs()
        return np.concatenate(([x_pos, x_vel], obs_original)).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        self.elapsed_steps = 0
        self.prev_x_pos = 0.0
        obs_original, info = self.env.reset(seed=seed, options=options)
        return self._get_obs(), info

    def step(self, action):
        obs_original, reward_original, terminated, truncated, info = self.env.step(action)
        self.elapsed_steps += 1

        x_pos = self.unwrapped_env.data.qpos[0]
        y_pos = self.unwrapped_env.data.qpos[1]

        # Recompensa por avance
        distance_moved = x_pos - self.prev_x_pos
        reward = distance_moved * 10.0

        # Penalización por salirse del carril
        if abs(y_pos) > 2.0:
            reward -= 0.5

        # Pequeña penalización por tiempo
        reward -= 0.01

        # Terminación: solo por meta o tiempo agotado (ignoramos caídas)
        done = False
        if x_pos >= self.goal_distance:
            tiempo_real = self.elapsed_steps * self.dt
            print(f"🎉 Meta alcanzada en paso {self.elapsed_steps} (tiempo: {tiempo_real:.1f} s)")
            reward += 100.0
            done = True
        elif self.elapsed_steps >= self.max_steps:
            tiempo_real = self.max_steps * self.dt
            print(f"⏰ Tiempo agotado: {tiempo_real:.1f} segundos, distancia recorrida: {x_pos:.1f} m")
            done = True

        self.prev_x_pos = x_pos
        obs = self._get_obs()
        return obs, reward, done, False, info

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()

# Registrar el entorno (con nombre único)
try:
    register(id='RacingAntReal-v0', entry_point='__main__:RacingAntEnv')
except (gym.error.NameNotFound, gym.error.Error):
    pass

# --- Preparación de directorios ---
if os.path.exists("./logs"):
    shutil.rmtree("./logs", ignore_errors=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./best_model", exist_ok=True)

# --- Entornos vectorizados ---
n_envs = 4
train_env = DummyVecEnv([lambda: RacingAntEnv(render_mode=None) for _ in range(n_envs)])
eval_env = RacingAntEnv(render_mode='human')   # ventana de evaluación

# --- Modelo PPO ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")

model = PPO(
    'MlpPolicy',
    train_env,
    verbose=0,
    device=device,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
)

# --- Callback con render cada 10k pasos ---
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./best_model",
    log_path="./logs",
    eval_freq=10_000,
    deterministic=True,
    render=True
)

# --- Entrenamiento (2M pasos para dar tiempo a aprender 100m) ---
print("🏁 Comenzando entrenamiento (100 metros en 60 segundos reales)...")
model.learn(total_timesteps=2_000_000, callback=eval_callback, progress_bar=True)
model.save("modelo_carrera_real")
print("✅ Entrenamiento completado.")

# --- Generar MP4 del mejor modelo ---
best_model_path = "./best_model/best_model.zip"
if os.path.exists(best_model_path):
    best_model = PPO.load(best_model_path)
    print("Cargado mejor modelo")
else:
    best_model = model
    print("No hay mejor modelo, se usa el final")

rec_env = RacingAntEnv(render_mode='rgb_array')
obs, _ = rec_env.reset()
frames = []
max_frames = 7000   # más de 60 segundos
for step in range(max_frames):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, _, done, _, _ = rec_env.step(action)
    frames.append(rec_env.render())
    if done:
        print(f"Episodio terminado en paso {step+1}")
        break
rec_env.close()

print(f"Frames capturados: {len(frames)}")
if frames:
    mp4_path = "carrera_real.mp4"
    writer = imageio.get_writer(mp4_path, fps=30, format='ffmpeg', codec='libx264')
    for frame in frames:
        writer.append_data(frame)
    writer.close()
    print(f"✅ Vídeo guardado: {mp4_path}")
else:
    print("No se capturaron frames.")

Output()

c:\Users\Carla\AppData\Local\Programs\Python\Python313\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment RacingAntReal-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


dt=0.05 seg/paso → 1200 pasos para 60 segundos
dt=0.05 seg/paso → 1200 pasos para 60 segundos
dt=0.05 seg/paso → 1200 pasos para 60 segundos
dt=0.05 seg/paso → 1200 pasos para 60 segundos
dt=0.05 seg/paso → 1200 pasos para 60 segundos
Dispositivo: cpu
🏁 Comenzando entrenamiento (100 metros en 60 segundos reales)...


⏰ Tiempo agotado: 60.0 segundos, distancia recorrida: 0.9 m

⏰ Tiempo agotado: 60.0 segundos, distancia recorrida: 3.1 m

...

⏰ Tiempo agotado: 60.0 segundos, distancia recorrida: -0.3 m

⏰ Tiempo agotado: 60.0 segundos, distancia recorrida: 2.2 m

⏰ Tiempo agotado: 60.0 segundos, distancia recorrida: 4.9 m

⏰ Tiempo agotado: 60.0 segundos, distancia recorrida: 1.9 m

✅ Entrenamiento completado.
Cargado mejor modelo
dt=0.05 seg/paso → 1200 pasos para 60 segundos
⏰ Tiempo agotado: 60.0 segundos, distancia recorrida: 2.8 m
Episodio terminado en paso 1200
Frames capturados: 1200
✅ Vídeo guardado: carrera_real.mp4


In [ ]:
env = RacingAntEnv(render_mode='rgb_array')
obs, _ = env.reset()
frames = []
max_steps = 2000

print("Generando frames...")
for step in range(max_steps):
    action, _ = model.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = env.step(action)
    frames.append(env.render())
    if terminated or truncated:
        print(f"Carrera terminada tras {step+1} pasos.")
        break
env.close()

# --- 4. Guardar como MP4 (requiere ffmpeg) ---
print(f"Se capturaron {len(frames)} frames. Guardando MP4...")
mp4_path = "carrera_ant.mp4"
writer = imageio.get_writer(mp4_path, fps=30, format='ffmpeg', codec='libx264')
for frame in frames:
    writer.append_data(frame)
writer.close()
print(f"✅ Vídeo guardado como {mp4_path}")

: 